<a href="https://colab.research.google.com/github/justdoit0430/newcrawl/blob/main/news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import requests
import json
from datetime import datetime, timedelta
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from urllib.parse import quote, unquote
import re
import time
import logging
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files # Colab 파일 다운로드용

# 1. 로깅 설정 (Colab에서는 출력이 바로 보임)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 2. 네이버 API 설정 (보안이 걱정되신다면 Colab 왼쪽 '열쇠' 아이콘(Secrets)에 저장 후 사용하세요)
client_id = "W82Xhl54o5knFsACcgV6"
client_secret = "4oRhzO3DqM"

headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret
}

# 날짜 설정
current_date = datetime.now()
one_year_ago = current_date - timedelta(days=365)

final_news_list = []
seen_titles = set() # 중복 제거용

def clean_html(raw_html):
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', raw_html)
    cleantext = unquote(cleantext)
    return cleantext

def format_date(date_str):
    try:
        return datetime.strptime(date_str, '%a, %d %b %Y %H:%M:%S %z').strftime('%Y-%m-%d')
    except:
        return datetime.now().strftime('%Y-%m-%d')

def fetch_news_for_keyword(keyword, retries=3):
    encoded_search_word = quote(keyword)
    local_news_list = []

    # 네이버 API 최대 허용치인 1000개까지 조회
    for start in range(1, 1001, 100):
        api_url = (
            f"https://openapi.naver.com/v1/search/news.json"
            f"?query={encoded_search_word}&start={start}&display=100&sort=date"
        )
        try:
            response = requests.get(api_url, headers=headers, timeout=10)
            if response.status_code == 429: # Rate Limit 발생 시 잠시 대기
                time.sleep(2)
                continue
            response.raise_for_status()
        except Exception as e:
            if retries > 0:
                time.sleep(1)
                return fetch_news_for_keyword(keyword, retries - 1)
            logging.error(f"'{keyword}' 에러: {e}")
            return local_news_list

        news_data = response.json()
        if not news_data.get('items'):
            break

        for item in news_data['items']:
            news_date_str = format_date(item['pubDate'])
            news_date = datetime.strptime(news_date_str, '%Y-%m-%d').date()

            if news_date >= one_year_ago.date():
                news_title = clean_html(item['title'])
                news_link = item['link']

                if news_title not in seen_titles:
                    seen_titles.add(news_title)
                    local_news_list.append([keyword, news_date_str, news_title, news_link])
            else:
                return local_news_list
    return local_news_list

def fetch_news_parallel(keywords):
    # Colab 사양에 따라 max_workers 조절 (5~10 권장)
    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_keyword = {executor.submit(fetch_news_for_keyword, keyword): keyword for keyword in keywords}
        for future in as_completed(future_to_keyword):
            keyword = future_to_keyword[future]
            try:
                news_items = future.result()
                final_news_list.extend(news_items)
                logging.info(f"✅ '{keyword}' 수집 완료: {len(news_items)}건")
            except Exception as e:
                logging.error(f"❌ '{keyword}' 실패: {e}")

# 키워드 목록
search_keywords = "유미코아","GM","실리콘음극재","전고체","차세대 전지","르노","CATL","삼성SDI","LG에너지솔루션","SK온","파나소닉","LG화학","토요타","스텔란티스","BMW","FEOC", "포스코퓨처엠","포스코", "양극재","음극재","트럼프","거시경제", "엘앤에프","에코프로", "전구체", "전기차", "ESS", "데이터센터","전력"

# 수집 실행
print(f"🚀 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} 뉴스 수집을 시작합니다.")
fetch_news_parallel(search_keywords)

# 데이터 가공
df = pd.DataFrame(final_news_list, columns=["Keyword", "Date", "Title", "Link"])

if not df.empty:
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by='Date', ascending=False)
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

    # 파일 저장 (Colab 세션 내 저장)
    file_name = f"battery_news_1year_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    df.to_excel(file_name, index=False)

    # 하이퍼링크 스타일 적용
    try:
        wb = load_workbook(file_name)
        ws = wb.active
        for row in range(2, ws.max_row + 1):
            title_cell = ws.cell(row=row, column=3)
            link = ws.cell(row=row, column=4).value
            if link:
                title_cell.hyperlink = link
                title_cell.font = Font(color='0000FF', underline='single')
        wb.save(file_name)
        print(f"\n파일 생성이 완료되었습니다: {file_name}")

        # --- Colab 전용: 파일 자동 다운로드 ---
        files.download(file_name)
    except Exception as e:
        print(f"엑셀 편집 중 오류 발생: {e}")
else:
    print("수집된 데이터가 없습니다.")

🚀 2026-03-23 00:11:17 뉴스 수집을 시작합니다.

파일 생성이 완료되었습니다: battery_news_1year_20260323_001134.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>